# The

In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

## Grab data from csv, import to dataframe, split into features and labels

In [2]:
df = pd.read_csv("creditcard.csv")

X = df.loc[:, df.columns != "Class"]
y = df["Class"]

X, y


(            Time         V1         V2        V3        V4        V5  \
 0            0.0  -1.359807  -0.072781  2.536347  1.378155 -0.338321   
 1            0.0   1.191857   0.266151  0.166480  0.448154  0.060018   
 2            1.0  -1.358354  -1.340163  1.773209  0.379780 -0.503198   
 3            1.0  -0.966272  -0.185226  1.792993 -0.863291 -0.010309   
 4            2.0  -1.158233   0.877737  1.548718  0.403034 -0.407193   
 ...          ...        ...        ...       ...       ...       ...   
 284802  172786.0 -11.881118  10.071785 -9.834783 -2.066656 -5.364473   
 284803  172787.0  -0.732789  -0.055080  2.035030 -0.738589  0.868229   
 284804  172788.0   1.919565  -0.301254 -3.249640 -0.557828  2.630515   
 284805  172788.0  -0.240440   0.530483  0.702510  0.689799 -0.377961   
 284806  172792.0  -0.533413  -0.189733  0.703337 -0.506271 -0.012546   
 
               V6        V7        V8        V9  ...       V20       V21  \
 0       0.462388  0.239599  0.098698  0.36378

## Remove bad values and check for empty values

In [3]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

na_x = X.isna().sum(axis=0)
na_y = y.isna().sum(axis=0)

print(na_x, na_y)

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
dtype: int64 0


## Subtract mean, divide by stdev

In [ ]:
X -= np.mean(X,axis=0)
X /= np.std(X,axis=0)

for i in range(len(y)):
    if y[i] == 0:
        y[i] = 1
    else:
        y[i] = 0

X = X.values.astype('float64')
y = y.values

array([1, 1, 1, ..., 1, 1, 1], shape=(284807,))

## Convert to Torch tensors

In [5]:
X = torch.tensor(X).float()
y = torch.tensor(y).long()

## Convert to tensor dataset, train test split

In [6]:
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import random_split
from torchmetrics.functional.classification import binary_confusion_matrix

train_data, test_data = random_split(TensorDataset(X, y), [0.8, 0.2])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

/Users/ahadj/Desktop/25-26/Winter Quarter/CSC 487/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model Architecture

In [7]:
linear_model = torch.nn.Sequential(
    torch.nn.Linear(30, 256),
    torch.nn.GELU(),
    torch.nn.Linear(256, 64),
    torch.nn.GELU(),
    torch.nn.Linear(64, 2),
)

loss_fn = torch.nn.CrossEntropyLoss()
lr = 3e-4
opt = torch.optim.AdamW(linear_model.parameters(), lr=lr)

In [8]:
for epoch in range(10):
    for batch in tqdm(train_loader, desc=f'Epoch: {epoch}'):
        X_batch, y_batch = batch
        pred = linear_model(X_batch)
        loss = loss_fn(pred, y_batch)
        opt.zero_grad()
        loss.backward()
        opt.step()

Epoch: 9: 100%|██████████| 3561/3561 [00:03<00:00, 1131.78it/s]


In [9]:
linear_model.eval()

correct = 0
total = 0

cf = torch.zeros(2,2)

with torch.no_grad():
    for X_batch, y_batch in tqdm(test_loader, desc=f"Predicting..."):
        pred = linear_model(X_batch)
        predicted_labels = pred.argmax(dim=1)
        cf += binary_confusion_matrix(predicted_labels, y_batch)
        correct += (predicted_labels == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct/total
print(accuracy)

Predicting...: 100%|██████████| 891/891 [00:00<00:00, 3550.61it/s]

0.9993679886237953


In [11]:
tp = cf[0][0]
fn = cf[0][1]
fp = cf[1][0]
tn = cf[1][1]
recall = tp / (tp + fn)
precision = tp / (tp + fp)
f1 = (2 * precision * recall) / (precision + recall)
print(f"true positives: {tp} \ntrue negatives: {tn} \nfalse positives: {fp} \nfalse negatives: {fn} \nprecision: {precision} \nrecall: {recall}")

true positives: 85.0 
true negatives: 56840.0 
false positives: 6.0 
false negatives: 30.0 
precision: 0.9340659379959106 
recall: 0.739130437374115
